# LLM'in Sınırı

- Eğitim verisi sonrası bilgi yok.

- Hallucination -- emin gibi yalan söyler

- Kendi şirket verini bilmez

- Güncel haber/fiyat/stok bilgisi yok

- Gizli döküman okuyamaz

    - Çözüm RAG

# RAG -- TANIM

- Retrieval-Augmented Generation

- Cevap üretmeden önce ilgili bilgiyi getir, context olarak LLM'e ver

1. 3 adım:

- Retrieve: Sorguya ilgili docs bul (vektör DB'den)

- Augment: Docs'ı prompt'a ekle

- Generate: LLM cevabı üretir

# RAG Pipeline

- Kullanıcı Sorusu -> Embedding -> Vektör DB Arama -> Top-K Docs -> LLM + Context -> Cevap

# İnşa Zamanı

- Indexing (bir kerelik)
    - Load -> Split -> Embed -> Store

- Query Time (her soruda)
    - Embed query -> Search -> Retrieve -> Prompt -> LLM -> Answer
    

# Koşan Örnek: Sepetix Destek Asistanı

- Sepetix = kurgusal bir e-ticaret sitesi (gerçek değil)

- LLM bu sirketi bilmez - kargo eşiği, iade süresi , üyelik fiyatı eğitim verisinde yok

1. Bu modül boyunca tek bir hedef:

- Müşteri sorularını Sepetix'in kendi belgelerinden cevaplayan bir asistan
- Cevabın hangi belgeden geldiğini söyler (kaynak gösterimi)
- Asistanı adım adım kuracağız: Load -> Split -> Store -> Retrieve

    - Örnek gerçekler: ücretsiz kargo 250 TL üzeri/ 14 gün / Sepetix Plus yıllık 200 TL gibi

# NEDEN RAG? - RAG'siz vs Rag'li

1. Soru: "Sepetix Plus yıllık kaç TL"

- RAG'siz
    - Model Sepetix'i hiç görmedi -> bilemez ya da güvende uydurur ("149 TL gibi...") 
    - Kaynak yok doğrulayamaz/ halusinasyon riski

- RAG'li

    - İlgili belgeyi getirir -> 199 TL + Kaynak: Üyelik & Sadakat

    - Belgeeye dayalı / dogrulanabilir / güncel

# rag vs Fine-tuning
-                 RAG    --- Fine Tuning
- Yeni Veri     ------        Kolay Ekle      ----     Tekrar eğit  
- Güncellik     ------         Anlık        ----     Eğitim zamanında donmuş 
- Halusinasyon    -------        Az        -----        Hala Olabilir  
- Maliyet      -----------     Düşük      -----          Yüksek (GPU)
- Style/tone   -------------   Sınırlı      ----             Güçlü  

# RAG vs Uzun Bağlam (2026)

- Modeller artık 1M+ token bağlam alıyor. "RAG öldü mü?" -> Hayır, ama ne zaman kullanılacağı

- Küçük(~<200K token / ~500 sayfa) ve statik içerik -> hepsini doğrudan bağlamda koy, RAG gereksiz
- Büyük / sık değişen, maliyet-gecikme hassas veya kaynak-gösteri gereken -> RAG
- 2026 kazananı genelde HİBRİT: retrieval ile en alakalı ~50-200K token'a indir, sonra uzun-bağlam modeline ver

# Özet 

- RAG = Retrieve + Augment + Generate

- LLM'in güncellik + hallucination problemi çözer

- Indexing (bir kez) + Query(her soruda)

- RAG >> Fine-tuning (güncellik için)

In [ ]:
import os 
# pip install langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter



# 1. Şirket belgelerini yukle

- İçerik iki formatta: Markdown politika belgeler + bir PDF müşteri rehberi (gerçek hayatta kaynaklar karışık formattadır.)

In [ ]:
md_docs = DirectoryLoader("şirket-bilgi-bankasi/", glob = "**/*.md", loader_cls = TextLoader().load())
print(f"{len(md_docs)} Markdown belge:")
for d in md_docs:
    print(" -", os.path.basename(d.metadata["source"])) 
    print(" -", d.metadata)
    print(" -", d.page_content[:100])  

In [ ]:
pdf_docs = PyPDFLoader("şirket-bilgi-bankasi/sepetix-rehber.pdf").load()
print(f"PDF: {len(pdf_docs)} sayfa")
print("İlk 150 karakter:\n", pdf_docs[0].page_content[:150])

## Farklı formatlar da yuklenebilir (CSV, web,Word)

- RAG kaynakları çoğu zaman karışık formatlı olur. Örneğin Sepetix'in ürün listesi bir CSV -- her satır bir Document olur (yapısal veri) (Not: chatbot corpus'umuz yime .md belgeleri; bu hucre sadece loader çeşitliliği gösterir.)

In [ ]:
# CSV - her satır bir Document (yapısal veri)
urun_docs = CSVLoader("sirket-bilgi-bankasi/urunler.csv").load()
print(f"CSV: {len(urun_docs)} satır -> {len(urun_docs)} Document")
print(urun_docs[0].page_content)

# Web sayfasından yükleme (ağ ister)

- Bir web sayfasından doğrudan çekebilirsin. Örnek oalrak Türkçe Wikipedia'nın e-tiaret sayfasını alalım (sunucu-render'lı sayfalar temiz gelir).

In [ ]:
import os; os.environ["USER_AGENT"] = "c2w-lesson/1.0"
from langchain_community.document_loaders import WebBaseLoader

web_docs = WebBaseLoader("https://tr.wikipedia.org/wiki/E-ticaret").load()
print(f"Web: {len(web_docs[0].page_content):,} karakter çekildi")
print(web_docs[0].page_content)
# Web çıktısı menu/gürültü de içerir -> pratikte temizlenir (chunk öncesi)

# Text Splitting -- Neden ?

- LLM context sınırı: 8k-128 token max

- Retrieval verimlilik: küçük chunk -> daha odaklı cevap

- embedding boyutu: uzun text embedding'i zayıflar

- Maliyer: daha küçük prompt -> daha az token

# 2. Neden chunk'lara böleriz?

- Tüm belgeleri tek seferde model veremeiz (context sınırı) - ve 'iade süresi' sorusuna tüm belgeler değil ilgili paragraf lazım. Bu yizden küçük chunk'lara böleriz; sınırda bilgi kaybını önlemek için biraz overlap bırakırız.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 60)
chunks = splitter.split_documents(md_docs)
print(f" {len(md_docs)} belge -> {len(chunks)} chunks\n")

# İlk 3 chunk'ın İÇİNE bakalım - böldükten sonra ne oluştu?

for i, ch in enumerate(chunks[:3]):
    kaynak = os.path.basename(ch.metadata["source"])
    print(f"--- Chunk {i}   .   {len(ch.page_content)} karakter     .   {kaynak} ---")
    print(ch.page_content, "\n")

- Tüm chunk'lara kuşbakışı: uzunluklar chuck_size'a (400) yakın ama birebir değil - splitter cümle/başlık sınırlarına saygı duyar, ortadan kesmez.

In [ ]:
for i, ch in enumerate(chunks):
    kaynak = os.path.basename(ch.metadata["source"])
    print(f"{i:2d}   {len(ch.page_content):3d} karakter {kaynak}")

# chunk_size & overlap

- chunk_size ~400-1000 karakter(küçük -> bağlam kaybı, büyük -> gürültü)
- overlap = chunk_size'ın ~%10-20'si
- Başla: RecursiveCharacterTextSplitter(400,60), retreival kalitesine göre ayarla

In [ ]:
KATEGORİ = {
    "kargo-teslimat": "Kargo & Teslimat", "iade-degisim": "İade % Degisim",
    "siparis-odeme": "Siparis & Ödeme", "uyelik-sadakat": "Üyelik & Sadakat",
    "sss": "Sıkça Sorulan Sorular"
}
for ch in chunks:
    ad = os.path.basename(ch.metadata["source"]).replace(".md","")
    ch.metadata("kategorisi:", chunks[0].metadata["kategori"])

print("chunk kategorisi:", chunks[0].metadata["kategori"])
print({c.metadata["kategori"] for c in chunks})

# 3. Markdown-aware bölme (başlık hiyerarşisini koru)

- Belgeler Markdown başlıklarıyla yazılı. MarkdownHeaderTestSplitter başlığı metadata'ya koyar -- yapılı belgelerde retrieval'ı iyileştirir.



In [ ]:
# 1) İade belgesini bul (kaynağında "aide geçen")
iade_belge = next(d for d in md_docs if "iade" in d.metadata["source"])

# 2) Başlıklara göre böl -> ## metadata'ya taşınır
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [("#","Belge"),("##","Başlık")]
)
bolumler = md_splitter.split_text(iade_belge.page_content)

# 3) Her bölüm hangi başlıktan geldi?
for bolum in bolumler:
    print(bolum.metadata, "->", bolum.page_content[:50].replace("\n"," "))
    

# Best Practices

- Metedata her zaman ekle(source, page, date)
- Başla RecursiveCharaterTextSplitter(400,60)
- A/B test: farklı chunk_size -> hangisi daha iyi retrieval
- Markdown'da header-aware splitter
- Kod için language-aware (Python,JS)